In [1]:
import pandas as pd
import numpy as np 
import sys 
import torch
sys.path.append('../utils/')
from mxnet_utils import *
from nlp_utils import *

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cuda


# Trying MXNet Structure

In [3]:
import pickle

with open("../data/vocabs.pkl", "rb") as f:
    vocabs = pickle.load(f)

topic_vocab       = vocabs["topic_vocab"]
author_vocab      = vocabs["author_vocab"]
job_vocab         = vocabs["job_vocab"]
location_vocab    = vocabs["location_vocab"]
affiliation_vocab = vocabs["affiliation_vocab"]
label_map         = vocabs["label_map"]

print("Loaded vocabs and label_map.")

Loaded vocabs and label_map.


In [4]:
import random
from transformers import BertModel, BertTokenizer, BertConfig

np.random.seed(100)
torch.manual_seed(100)
random.seed(100)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(100)

bert_dropout = 0.1 # From later in the notebook
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
config = BertConfig.from_pretrained('bert-base-uncased',
                                    hidden_dropout_prob=bert_dropout,
                                    attention_probs_dropout_prob=bert_dropout)
bert_base = BertModel.from_pretrained('bert-base-uncased', config=config)
bert_base.to(device)
print(bert_base)

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [5]:
def feature_extraction_transform(data):
    # Transform label into position / negative
    try:
        label, statement, topics, author, job, location, affiliation,cnt_barely, cnt_false, cnt_half, cnt_mostly, cnt_pants_on_fire, venue_context, justification = [str(e) for e in data[2:16]]
    except Exception as e:
        print(f"Error in feature_extraction_transform: {e}, data: {data}")
        return None # Return None to filter out this bad data

    topic_one_encoding = np.zeros(shape=(len(topic_vocab)), dtype=np.float32)
    topic_ids = [topic_vocab[t.strip()] for t in topics.lower().split(',')]
    topic_one_encoding[topic_ids] = 1
    if len(topic_ids) > 0 and topic_one_encoding.sum() > 0:
        topic_one_encoding /= topic_one_encoding.sum()

    author_id = author_vocab[author.lower()]
    job_id = job_vocab[job.lower()]
    location_id = location_vocab[location.lower()]
    affiliation_id = affiliation_vocab[affiliation.lower()]

    try:
        cnt_barely_f = float(cnt_barely)
        cnt_false_f = float(cnt_false)
        cnt_half_f = float(cnt_half)
        cnt_mostly_f = float(cnt_mostly)
        cnt_pants_on_fire_f = float(cnt_pants_on_fire)
    except ValueError:
        # Handle cases where conversion to float fails
        cnt_barely_f = cnt_false_f = cnt_half_f = cnt_mostly_f = cnt_pants_on_fire_f = 0.0

    cnt_total = cnt_barely_f + cnt_false_f + cnt_half_f + cnt_mostly_f + cnt_pants_on_fire_f

    if cnt_total > 0 :
        proportion = [cnt_barely_f / cnt_total,
                      cnt_false_f / cnt_total,
                      cnt_half_f / cnt_total,
                      cnt_mostly_f / cnt_total,
                      cnt_pants_on_fire_f / cnt_total]
    else:
        proportion = [0.0, 0.0, 0.0, 0.0, 0.0]

    cnt_uncertainty = 1.0 / (cnt_total + 1.0)
    history_of_truth = np.array(proportion + [cnt_uncertainty], dtype=np.float32)
    venue_feature = 0 # venue_feature = f(venue_context), keep it as a vector

    return (statement, topic_one_encoding, author_id, job_id, location_id, affiliation_id,
            history_of_truth, venue_feature, label_map[label])

# Load Test Data

In [7]:
test_dataset_raw = load_tsv_data('../data/liar-plus/test2.tsv')
test_dataset = [d for d in [feature_extraction_transform(data) for data in test_dataset_raw] if d is not None]
test_dataset_bert = [transform_fn(*sample) for sample in test_dataset]
test_data_torch = ListDataset(test_dataset_bert)
test_data = DataLoader(test_data_torch,
                       batch_size=16,
                       shuffle=False,
                       collate_fn=collate_fn,
                       num_workers=0,
                       pin_memory=False)

# New Truth Model Adapted From Ryan's

In [8]:
import torch
import torch.nn.functional as F
import numpy as np

def run_truth_model(
    net,
    tokenizer,
    article_text,
    topic_vec,
    author_id,
    job_id,
    loc_id,
    aff_id,
    history_vec,
    device="cpu"
):
    net.eval()

    # ---------------------------
    # 1. Tokenize text
    # ---------------------------
    encoding = tokenizer(
        article_text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512
    )

    input_ids = encoding["input_ids"].to(device)
    token_types = encoding.get("token_type_ids", torch.zeros_like(input_ids)).to(device)
    attention_mask = encoding["attention_mask"].to(device)

    # ---------------------------
    # 2. Convert metadata to tensors
    # ---------------------------

    # Topic vector: should be shape (1, num_topics)
    if isinstance(topic_vec, np.ndarray):
        topic_vec = torch.tensor(topic_vec, dtype=torch.float32).unsqueeze(0)
    if topic_vec.dim() == 1:
        topic_vec = topic_vec.unsqueeze(0)
    topic_vec = topic_vec.to(device)

    # Categorical IDs
    author_id = torch.tensor([author_id]).to(device)
    job_id = torch.tensor([job_id]).to(device)
    loc_id = torch.tensor([loc_id]).to(device)
    aff_id = torch.tensor([aff_id]).to(device)

    # History vector (6-dim)
    if isinstance(history_vec, np.ndarray):
        history_vec = torch.tensor(history_vec, dtype=torch.float32).unsqueeze(0)
    if history_vec.dim() == 1:
        history_vec = history_vec.unsqueeze(0)
    history_vec = history_vec.to(device)

    # ---------------------------
    # 3. Forward pass
    # ---------------------------
    with torch.no_grad():
        logits = net(
            input_ids,
            token_types,
            attention_mask,
            topic_vec,
            author_id,
            job_id,
            loc_id,
            aff_id,
            history_vec
        )

        probs = F.softmax(logits, dim=-1)
        pred_idx = probs.argmax(dim=-1).item()

    # ---------------------------
    # 4. Convert to label string
    # ---------------------------
    class_labels = [
        "false",
        "half-true",
        "mostly-true",
        "true",
        "barely-true",
        "pants-fire"
    ]
    pred_label = class_labels[pred_idx]

    return {
        "logits": logits,
        "probabilities": probs,
        "predicted_label": pred_label,
        "predicted_index": pred_idx
    }

In [9]:
label_map = {
    'false': 0,
    'half-true': 1,
    'mostly-true': 2,
    'true': 3,
    'barely-true': 4,
    'pants-fire': 5
}

net_best2 = BERTClassifier(
    bert=bert_base,
    num_topics=len(topic_vocab),
    num_authors=len(author_vocab),
    num_jobs=len(job_vocab),
    num_locations=len(location_vocab),
    num_affiliations=len(affiliation_vocab),
    num_classes=len(label_map),
    embed_dim=32,
    author_dropout=0.1,
    author_mlp_layers=2,
    author_mlp_hidden=192,
    history_dropout=0.1,
    history_mlp_layers=2,
    history_mlp_hidden=192)
net_best2.load_state_dict(torch.load('../checkpoints/best.pth', map_location=device))
net_best2.to(device)
test_outputs = [run_truth_model(net_best2, tokenizer, statement, topic, aid, jid, lid, afid, hist, device=device) for statement, topic, aid, jid, lid, afid, hist, _, _ in test_dataset]

In [10]:
from sklearn.metrics import classification_report

label_map_enc = {v:k for k,v in label_map.items()}
y_test = [label_map_enc[d[-1]] for d in test_dataset]
print(classification_report(y_test,  pd.DataFrame(test_outputs)['predicted_label'].str.lower(), zero_division=0))

              precision    recall  f1-score   support

 barely-true       0.89      0.15      0.26       208
       false       0.40      0.65      0.49       249
   half-true       0.52      0.32      0.39       212
 mostly-true       0.45      0.44      0.44       265
  pants-fire       0.67      0.48      0.56        92
        true       0.38      0.59      0.46       241

    accuracy                           0.44      1267
   macro avg       0.55      0.44      0.44      1267
weighted avg       0.53      0.44      0.43      1267



In [ ]:
test_df = open_data('../data/liar-plus/test2.tsv')
test_df = test_df[['statement','label']]
test_df['sent_transformer_label'] = pd.DataFrame(test_outputs)['predicted_label'].str.lower()

### Now that we get the same accuracy from the training notebook, we add the sentence transformer features

In [15]:
val_dataset_raw = load_tsv_data('../data/liar-plus/val2.tsv')
val_dataset = [
    d for d in (feature_extraction_transform(row) for row in val_dataset_raw)
    if d is not None
]
val_df = open_data('../data/liar-plus/val2.tsv')
val_df = val_df[['statement', 'label']]
preds = []
for (statement, topic, aid, jid, lid, afid, hist, _, _) in val_dataset:
    pred_label = run_truth_model(
        net_best2,
        tokenizer,
        statement,
        topic,
        aid,
        jid,
        lid,
        afid,
        hist,
        device=device
    )
    preds.append(pred_label)

val_df['sent_transformer_label'] = [p['predicted_label'] for p in preds]
display(val_df.head(2))

,statement,label,sent_transformer_label
0,We have less Americans working now than in the...,barely-true,mostly-true
1,"When Obama was sworn into office, he DID NOT u...",pants-fire,pants-fire


In [17]:
train_dataset_raw = load_tsv_data('../data/liar-plus/train2.tsv')
train_dataset = [
    d for d in (feature_extraction_transform(row) for row in train_dataset_raw)
    if d is not None
]
train_df = open_data('../data/liar-plus/train2.tsv')
train_df = train_df[['statement', 'label']]
train_df.dropna(subset=['label'], inplace=True)
preds = []
for (statement, topic, aid, jid, lid, afid, hist, _, _) in train_dataset:
    pred_label = run_truth_model(
        net_best2,
        tokenizer,
        statement,
        topic,
        aid,
        jid,
        lid,
        afid,
        hist,
        device=device
    )
    preds.append(pred_label)

train_df['sent_transformer_label'] = [p['predicted_label'] for p in preds]
display(train_df.head(2))

KeyboardInterrupt: 

# Spam Score

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import MinMaxScaler, PowerTransformer
import torch
import pandas as pd

spam_model_name = "mrm8488/bert-tiny-finetuned-sms-spam-detection"
spam_tokenizer = AutoTokenizer.from_pretrained(spam_model_name)
spam_model = AutoModelForSequenceClassification.from_pretrained(spam_model_name)
spam_model.eval()

def get_spam_scores(text_list, batch_size=16):
    scores = []
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]
        inputs = spam_tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding="max_length",
            max_length=512
        )
        with torch.no_grad():
            outputs = spam_model(**inputs)
            probs = torch.softmax(outputs.logits, dim=1)
            scores.extend(probs[:, 1].tolist())
    return scores

train_df['spam_score'] = get_spam_scores(train_df['statement'].tolist())
test_df['spam_score'] = get_spam_scores(test_df['statement'].tolist())
val_df['spam_score'] = get_spam_scores(val_df['statement'].tolist())

pt = PowerTransformer(method='yeo-johnson')
train_df['spam_score'] = pt.fit_transform(train_df[['spam_score']])
val_df['spam_score'] = pt.transform(val_df[['spam_score']])   
test_df['spam_score'] = pt.transform(test_df[['spam_score']])
spam_scaler = MinMaxScaler(feature_range=(0,10))
train_df['spam_score'] = spam_scaler.fit_transform(train_df[['spam_score']])
val_df['spam_score'] = spam_scaler.transform(val_df[['spam_score']])   
test_df['spam_score'] = spam_scaler.transform(test_df[['spam_score']])

# Political Affiliation

In [ ]:
import spacy
# spacy.cli.download("en_core_web_md")
datum = train_df.iloc[0]
nlp = spacy.load("en_core_web_md")
doc = nlp(datum['statement'])
doc.vector.shape
statistic_types = {'CARDINAL', 'PERCENT', 'MONEY', 'QUANTITY'}

def stat_counter(text):
    if not isinstance(text, str):
        return 0
    doc = nlp(text)
    counter = 0
    for ent in doc.ents: 
        if ent.label_ in statistic_types:
            counter += 1
    return counter

from rapidfuzz import fuzz
conservative_bigrams = pd.read_csv('../data/top_conservative_bigrams.csv')['bigram']
liberal_bigrams = pd.read_csv('../data/top_liberal_bigrams.csv')['bigram']
def match_counter(statement, bigram_list, threshold):
    stat = nlp(str(statement))
    word = [word.text.lower() for word in stat]
    bigram_coll = [''.join(word[i:i+2]) for i in range(len(word)-1)]
    matches = 0
    for bigram in bigram_coll:
        for check in bigram_list:
            if fuzz.ratio(bigram, check) >= threshold:
                matches += 1
                break

    return matches


train_df['statistic_count'] = train_df['statement'].apply(stat_counter)
train_df['conservative_bigram_count'] = train_df['statement'].apply(
    lambda x: match_counter(x, conservative_bigrams, threshold=70)
)
train_df['liberal_bigram_count'] = train_df['statement'].apply(
    lambda x: match_counter(x, liberal_bigrams, threshold=70)
)

val_df['statistic_count'] = val_df['statement'].apply(stat_counter)
val_df['conservative_bigram_count'] = val_df['statement'].apply(
    lambda x: match_counter(x, conservative_bigrams, threshold=70)
)
val_df['liberal_bigram_count'] = val_df['statement'].apply(
    lambda x: match_counter(x, liberal_bigrams, threshold=70)
)

test_df['statistic_count'] = test_df['statement'].apply(stat_counter)
test_df['conservative_bigram_count'] = test_df['statement'].apply(
    lambda x: match_counter(x, conservative_bigrams, threshold=70)
)
test_df['liberal_bigram_count'] = test_df['statement'].apply(
    lambda x: match_counter(x, liberal_bigrams, threshold=70)
)

count_features = ["statistic_count", "conservative_bigram_count", "liberal_bigram_count"]
scaler_counts = MinMaxScaler(feature_range=(0,10))
count_pt = PowerTransformer(method='yeo-johnson')
train_df[count_features] = count_pt.fit_transform(train_df[count_features])
val_df[count_features] = count_pt.transform(val_df[count_features])   
test_df[count_features] = count_pt.transform(test_df[count_features])
train_df[count_features] = scaler_counts.fit_transform(train_df[count_features])
val_df[count_features]   = scaler_counts.transform(val_df[count_features])
test_df[count_features]  = scaler_counts.transform(test_df[count_features])

# Sensationalism

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def emotional_intensity_vader(text):
    if not isinstance(text, str):
        return 0.0
    if len(text) == 0:
        return 0.0
    vs = analyzer.polarity_scores(text)
    return abs(vs['compound'])

for df in [train_df, test_df, val_df]:
    df['emotional_intensity'] = df['statement'].apply(emotional_intensity_vader)

vader_pt = PowerTransformer(method='yeo-johnson')
train_df['emotional_intensity'] = vader_pt.fit_transform(train_df[['emotional_intensity']])
val_df['emotional_intensity'] = vader_pt.transform(val_df[['emotional_intensity']])   
test_df['emotional_intensity'] = vader_pt.transform(test_df[['emotional_intensity']])
scaler_vader = MinMaxScaler(feature_range=(0,10))
train_df["emotional_intensity"] = scaler_vader.fit_transform(train_df[["emotional_intensity"]])
val_df["emotional_intensity"]   = scaler_vader.transform(val_df[["emotional_intensity"]])
test_df["emotional_intensity"]  = scaler_vader.transform(test_df[["emotional_intensity"]])

# Complete Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

X_train, y_train = train_df.drop(columns=['statement','label']), train_df['label']
X_test, y_test = test_df.drop(columns=['statement','label']), test_df['label']

le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

clf = RandomForestClassifier(n_estimators=200, max_depth=7)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

NameError: name 'test_df' is not defined

In [60]:
X_train

,sent_transformer_label,spam_score,statistic_count,conservative_bigram_count,liberal_bigram_count,emotional_intensity
0,0,0.726939,0.0,0.000000,0.000000,2.565681
1,3,0.027961,0.0,0.588235,0.000000,3.706897
2,3,0.074133,0.0,0.000000,0.000000,3.265599
3,5,0.724981,0.0,0.588235,0.869565,7.778120
4,2,0.011558,0.0,0.000000,0.869565,0.000000
...,...,...,...,...,...,...
10237,3,0.527201,0.0,0.000000,0.434783,7.703202
10238,3,0.074917,0.0,0.588235,0.869565,4.124589
10239,0,0.712227,0.0,0.588235,0.000000,6.012931
10240,2,0.041343,0.0,0.000000,0.000000,0.000000
